# Day 1 · LigandLab: 2D similarity and pharmacophores

Compare several ligands in **2D**, build a simple **3D pharmacophore**, then use it to rank a small library. Start from a built-in teaching set or provide your own structures as SMILES, SDF, MOL, MOL2, CSV or SMI. You can also draw several molecules in the sketcher. Run the numbered steps from top to bottom in a Google Colab CPU runtime. The code is folded into form steps and can be expanded by the instructor.

**Practical design:** Yu Zhu and Taufiq Rahman · Department of Pharmacology, University of Cambridge. Found an error? Contact [Yu Zhu](mailto:yz876@cam.ac.uk) at yz876@cam.ac.uk.

This notebook teaches the idea behind ligand based pharmacophore screening. It uses RDKit conformers, alignment and chemical features. It does **not** reproduce PharmaGist's search algorithm or establish biological activity. The example molecules are a teaching set, not experimentally validated hits.

In [ ]:
# @title Step 1. Install tools { display-mode: "form" }
%pip -q install 'rdkit>=2024.9' 'pandas>=2' 'matplotlib>=3.7' 'py3Dmol>=2.0' 'ipywidgets>=8,<9' 'ipyketcher==0.1.0'

In [ ]:
# @title Step 2. Start LigandLab { display-mode: "form" }
from pathlib import Path
import csv, io, json, math, zipfile
import numpy as np
from rdkit import Chem, DataStructs, RDConfig
from rdkit.Chem import AllChem, ChemicalFeatures, Draw, rdFingerprintGenerator, rdMolAlign
from IPython.display import display, HTML
import ipywidgets as widgets
from ipywidgets import Output
import py3Dmol

WORK = Path('/content/ligandlab')
WORK.mkdir(exist_ok=True)
FEATURE_FACTORY = ChemicalFeatures.BuildFeatureFactory(str(Path(RDConfig.RDDataDir) / 'BaseFeatures.fdef'))
FAMILIES = ('Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'PosIonizable', 'NegIonizable')
COLOURS = {'Donor':'#2b6cb0', 'Acceptor':'#d64141', 'Aromatic':'#9959a9',
           'Hydrophobe':'#d89922', 'PosIonizable':'#2b9b8a', 'NegIonizable':'#e26d37'}
DEMO_TRAINING = [
    ('4-methoxybenzamide', 'NC(=O)c1ccc(OC)cc1'),
    ('4-ethoxybenzamide', 'NC(=O)c1ccc(OCC)cc1'),
    ('4-hydroxybenzamide', 'NC(=O)c1ccc(O)cc1'),
    ('4-methylbenzamide', 'NC(=O)c1ccc(C)cc1'),
]
DEMO_LIBRARY = [
    ('benzamide', 'NC(=O)c1ccccc1'),
    ('3-methoxybenzamide', 'NC(=O)c1cccc(OC)c1'),
    ('4-fluorobenzamide', 'NC(=O)c1ccc(F)cc1'),
    ('4-aminobenzamide', 'NC(=O)c1ccc(N)cc1'),
    ('methyl benzoate', 'COC(=O)c1ccccc1'),
    ('anisole', 'COc1ccccc1'),
    ('acetamide', 'CC(N)=O'),
    ('phenol', 'Oc1ccccc1'),
    ('pyridine', 'c1ccncc1'),
    ('cyclohexane', 'C1CCCCC1'),
]

def labelled_molecule(name, mol):
    if mol is None or mol.GetNumHeavyAtoms() == 0:
        raise ValueError(f'{name}: the molecule could not be read')
    if mol.GetNumHeavyAtoms() > 80:
        raise ValueError(f'{name}: more than 80 heavy atoms. Use a small molecule.')
    mol.SetProp('_Name', name)
    return name, mol

def parse_smiles_lines(text):
    items = []
    for number, line in enumerate(text.replace(' | ', '\n').splitlines(), 1):
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        parts = line.split()
        if len(parts) == 1:
            name, smiles = f'Ligand {number}', parts[0]
        else:
            smiles, name = parts[0], ' '.join(parts[1:])
        items.append(labelled_molecule(name, Chem.MolFromSmiles(smiles)))
    return items

def read_molecules(filename, data):
    suffix = Path(filename).suffix.lower()
    if suffix == '.sdf':
        supplier = Chem.ForwardSDMolSupplier(io.BytesIO(data), removeHs=False)
        return [labelled_molecule(m.GetProp('_Name') if m.HasProp('_Name') and m.GetProp('_Name') else f'Ligand {i}', m)
                for i, m in enumerate(supplier, 1) if m is not None]
    if suffix == '.mol':
        return [labelled_molecule(Path(filename).stem, Chem.MolFromMolBlock(data.decode(), removeHs=False))]
    if suffix == '.mol2':
        return [labelled_molecule(Path(filename).stem, Chem.MolFromMol2Block(data.decode(), removeHs=False))]
    if suffix == '.smi':
        return parse_smiles_lines(data.decode())
    if suffix == '.csv':
        reader = csv.DictReader(io.StringIO(data.decode()))
        if not reader.fieldnames or 'smiles' not in [c.lower() for c in reader.fieldnames]:
            raise ValueError('CSV needs a smiles column and may also have a name column')
        keys = {k.lower(): k for k in reader.fieldnames}
        return [labelled_molecule(row.get(keys.get('name'), '') or f'Ligand {i}',
                                  Chem.MolFromSmiles(row[keys['smiles']]))
                for i, row in enumerate(reader, 1)]
    raise ValueError('Use SDF, MOL, MOL2, SMI or CSV')

def make_conformers(mol, count=8, seed=17):
    result = Chem.AddHs(Chem.Mol(mol), addCoords=True)
    if result.GetNumConformers() and result.GetConformer().Is3D():
        return [result], 'Supplied 3D'
    result.RemoveAllConformers()
    params = AllChem.ETKDGv3()
    params.randomSeed = seed
    params.pruneRmsThresh = 0.35
    ids = list(AllChem.EmbedMultipleConfs(result, numConfs=count, params=params))
    if not ids:
        raise ValueError(f"Could not build 3D for {mol.GetProp('_Name')}")
    if AllChem.MMFFHasAllMoleculeParams(result):
        energies = AllChem.MMFFOptimizeMoleculeConfs(result, numThreads=1, maxIters=200)
    else:
        energies = AllChem.UFFOptimizeMoleculeConfs(result, numThreads=1, maxIters=200)
    conformers = []
    for cid, _ in sorted(zip(ids, energies), key=lambda pair:pair[1][1]):
        one = Chem.Mol(result)
        conf = Chem.Conformer(result.GetConformer(cid))
        one.RemoveAllConformers()
        one.AddConformer(conf, assignId=True)
        conformers.append(one)
    return conformers, 'Generated 3D'

def align_best(mol, reference, count=8):
    conformers, source = make_conformers(mol, count=count)
    best = None
    for conformer in conformers:
        try:
            alignment = rdMolAlign.GetO3A(conformer, reference)
            score = float(alignment.Score())
            alignment.Align()
            if best is None or score > best[0]:
                best = (score, conformer)
        except Exception:
            continue
    if best is None:
        raise ValueError(f"Could not align {mol.GetProp('_Name')}. Check its chemistry.")
    return best[1], round(best[0], 2), source

def features(mol):
    answer = []
    for feature in FEATURE_FACTORY.GetFeaturesForMol(mol):
        if feature.GetFamily() in FAMILIES:
            # BaseFeatures adds a hydrophobe to many aromatic carbons. The
            # aromatic ring centre already describes that part of this model.
            if feature.GetFamily() == 'Hydrophobe' and all(
                mol.GetAtomWithIdx(i).GetIsAromatic() for i in feature.GetAtomIds()):
                continue
            p = feature.GetPos()
            answer.append({'family':feature.GetFamily(), 'xyz':np.array([p.x,p.y,p.z], dtype=float)})
    return answer

def consensus_features(aligned, tolerance, min_support):
    per_ligand = [features(mol) for _, mol in aligned]
    output = []
    for anchor in per_ligand[0]:
        points = [anchor['xyz']]
        members = [aligned[0][0]]
        for (name, _), ligand_features in zip(aligned[1:], per_ligand[1:]):
            matches = [(np.linalg.norm(candidate['xyz']-anchor['xyz']), candidate)
                       for candidate in ligand_features if candidate['family'] == anchor['family']]
            if matches:
                distance, candidate = min(matches, key=lambda item:item[0])
                if distance <= tolerance:
                    points.append(candidate['xyz'])
                    members.append(name)
        if len(points) < min_support:
            continue
        centre = np.mean(points, axis=0)
        if any(old['family'] == anchor['family'] and
               np.linalg.norm(old['xyz']-centre) < 0.8 for old in output):
            continue
        output.append({'family':anchor['family'], 'xyz':centre,
                       'support':len(points), 'members':members,
                       'spread':float(np.sqrt(np.mean([np.sum((p-centre)**2) for p in points])))})
    output.sort(key=lambda feature:(-feature['support'], feature['family']))
    return output

print('LigandLab is ready. Start with the 2D ligands in Step 4.')

---

## 1 · Start with several ligands

Ligand based virtual screening starts with molecules of interest. First compare their **2D structures**, then ask whether they can share chemical features in 3D. We need several ligands to look for agreement. The demonstration compounds are related model molecules, not a validated active series.

Use the demonstration first. For your own data, supply at least three related ligands. You can upload several files together. An SDF can contain several molecules. A `.smi` file should have one `SMILES name` pair per line. A CSV needs a `smiles` column and can have a `name` column. In the SMILES form field, separate entries with ` | `, for example `CCO ethanol | CCN ethylamine | CCC propane`. Existing 3D coordinates in SDF, MOL or MOL2 are retained. Check protonation, stereochemistry and bond orders before interpretation.

In [ ]:
# @title Step 3. Sketch ligands (optional) { display-mode: "form" }
show_sketcher = False # @param {type:'boolean'}
drawn_ligands = globals().get('drawn_ligands', [])
if show_sketcher:
    from google.colab import output
    output.enable_custom_widget_manager()
    import traitlets
    try:
        traitlets.List[str]
    except TypeError:
        traitlets.List.__class_getitem__ = classmethod(lambda cls, item: cls)
    from ipyketcher import KetcherEditor
    sketcher = KetcherEditor(return_formats=['smiles'])
    drawing_name = widgets.Text(value='', placeholder='Ligand name', description='Name')
    add_button = widgets.Button(description='Add this drawing', button_style='success')
    reset_button = widgets.Button(description='Clear saved drawings')
    status = widgets.Output()
    def add_drawing(_):
        with status:
            status.clear_output(wait=True)
            smile = (sketcher.smiles or '').strip()
            if not smile or Chem.MolFromSmiles(smile) is None:
                print('Draw a valid molecule first.')
                return
            name = drawing_name.value.strip() or f'Drawn ligand {len(drawn_ligands)+1}'
            drawn_ligands.append((name, smile))
            print(f'Saved {name}. Draw another molecule, or run Step 4.')
            print('Saved drawings:', ', '.join(n for n, _ in drawn_ligands))
    def clear_drawings(_):
        drawn_ligands.clear()
        with status:
            status.clear_output(wait=True)
            print('Saved drawings cleared.')
    add_button.on_click(add_drawing)
    reset_button.on_click(clear_drawings)
    display(sketcher, widgets.HBox([drawing_name, add_button, reset_button]), status)
else:
    print('Tick show_sketcher to draw several ligands. Save each drawing with the button, then choose Drawings in Step 4.')

In [ ]:
# @title Step 4. Choose training ligands { display-mode: "form" }
training_source = 'Demo' # @param ['Demo', 'SMILES list', 'Upload file', 'Drawings']
training_smiles = '' # @param {type:'string'}
if training_source == 'Demo':
    training = [labelled_molecule(name, Chem.MolFromSmiles(smiles)) for name, smiles in DEMO_TRAINING]
elif training_source == 'SMILES list':
    training = parse_smiles_lines(training_smiles)
elif training_source == 'Drawings':
    training = [labelled_molecule(name, Chem.MolFromSmiles(smiles)) for name, smiles in drawn_ligands]
else:
    from google.colab import files
    print('Upload one or more SDF, MOL, MOL2, SMI or CSV files containing the training ligands')
    uploaded = files.upload()
    if not uploaded:
        raise ValueError('Upload at least one file')
    training = [item for filename, data in uploaded.items()
                for item in read_molecules(filename, data)]
if not 3 <= len(training) <= 12:
    raise ValueError('Use 3 to 12 training ligands for this practical')
smiles_set = {Chem.MolToSmiles(Chem.RemoveHs(mol), isomericSmiles=True) for _, mol in training}
if len(smiles_set) != len(training):
    raise ValueError('Training ligands must be different molecules')
if len({name for name, _ in training}) != len(training):
    raise ValueError('Give each training ligand a different name')
picture = Draw.MolsToGridImage([Chem.RemoveHs(mol) for _, mol in training],
                               legends=[name for name, _ in training], molsPerRow=3, subImgSize=(270,220))
display(picture)
print(f'{len(training)} training ligands loaded. These 2D drawings do not specify their 3D shapes.')

### Compare the 2D structures

A **Morgan fingerprint** records local atom environments as bits. Here we use a radius of 2 bonds and 2,048 bits, with stereochemistry included when specified. The **Tanimoto score** is the number of bits present in both fingerprints divided by the number present in either fingerprint. It runs from 0 to 1. A value of 1 means identical fingerprints under these settings, which does not always mean identical molecules.

The heatmap compares every pair of training ligands. The cut-off below is only a visual aid for discussion, not an activity threshold. Notice which pairs are most similar before you build the 3D pharmacophore.

In [ ]:
# @title Step 5. Compare 2D similarity { display-mode: "form" }
similarity_cutoff = 0.50 # @param {type:'number'}
if not 0 <= similarity_cutoff <= 1:
    raise ValueError('The Tanimoto cut-off must be between 0 and 1')
import pandas as pd
import matplotlib.pyplot as plt
MORGAN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048, includeChirality=True)
def fingerprint(mol):
    return MORGAN.GetFingerprint(Chem.RemoveHs(mol))
training_fingerprints = [fingerprint(mol) for _, mol in training]
names = [name for name, _ in training]
similarity_matrix = np.array([
    [DataStructs.TanimotoSimilarity(first, second) for second in training_fingerprints]
    for first in training_fingerprints])
similarity_table = pd.DataFrame(similarity_matrix, index=names, columns=names)
similarity_pairs = sorted([
    {'ligand A':names[i], 'ligand B':names[j], 'Tanimoto':round(float(similarity_matrix[i,j]),3)}
    for i in range(len(names)) for j in range(i+1,len(names))],
    key=lambda row:-row['Tanimoto'])
fig, ax = plt.subplots(figsize=(max(6,len(names)*0.8), max(5,len(names)*0.7)))
heatmap = ax.imshow(similarity_matrix, vmin=0, vmax=1, cmap='YlGnBu')
ax.set_xticks(range(len(names)), names, rotation=45, ha='right')
ax.set_yticks(range(len(names)), names)
for i in range(len(names)):
    for j in range(len(names)):
        value = similarity_matrix[i,j]
        ax.text(j,i,f'{value:.2f}',ha='center',va='center',
                color='white' if value >= 0.55 else '#172b3a',fontsize=9)
fig.colorbar(heatmap,ax=ax,label='Tanimoto similarity')
ax.set_title('2D Morgan fingerprint similarity')
fig.tight_layout()
fig.savefig(WORK/'tanimoto_heatmap.png', dpi=180, bbox_inches='tight')
plt.show()
display(pd.DataFrame(similarity_pairs))
print(f"{sum(row['Tanimoto'] >= similarity_cutoff for row in similarity_pairs)} of {len(similarity_pairs)} distinct pairs meet the chosen cut-off.")

### Calculate the score for any pair

Choose two ligands below. Their 2D structures appear side by side above the calculation. The coloured bar separates bits found only in A, bits found in both, and bits found only in B. The shared bits form the numerator. All three groups together form the union in the denominator.

$$T(A,B)=\frac{|F_A \cap F_B|}{|F_A \cup F_B|} = \frac{\text{shared bits}}{\text{bits in either fingerprint}}$$

Each molecule is represented by 2,048 numbered fingerprint slots, from 0 to 2,047. A Morgan fingerprint sets a slot when it finds a local atom environment. For example, bit 322 means that slot 322 is switched on. It is not atom 322, an atom count, or a position in the 2D drawing. Hashing can put different environments in the same slot, so a shared bit does not by itself prove that the molecules contain exactly the same fragment, shape or activity. The optional lists below show the switched-on slot numbers for each group.

In [ ]:
# @title Step 6. Calculate a Tanimoto score { display-mode: "form" }
from IPython.display import Math, clear_output
pair_a = widgets.Dropdown(options=names, value=similarity_pairs[0]['ligand A'],
                          description='Ligand A', layout=widgets.Layout(width='350px'))
pair_b = widgets.Dropdown(options=names, value=similarity_pairs[0]['ligand B'],
                          description='Ligand B', layout=widgets.Layout(width='350px'))
show_bit_numbers = widgets.Checkbox(value=False, description='Show fingerprint slot numbers')
calculation_output = widgets.Output()

def show_tanimoto_calculation(change=None):
    with calculation_output:
        clear_output(wait=True)
        index_a = names.index(pair_a.value)
        index_b = names.index(pair_b.value)
        structures = Draw.MolsToGridImage(
            [Chem.RemoveHs(training[index_a][1]), Chem.RemoveHs(training[index_b][1])],
            legends=[f'A: {pair_a.value}', f'B: {pair_b.value}'],
            molsPerRow=2, subImgSize=(340, 240))
        display(structures)
        bits_a = set(training_fingerprints[index_a].GetOnBits())
        bits_b = set(training_fingerprints[index_b].GetOnBits())
        a_only = bits_a - bits_b
        shared = bits_a & bits_b
        b_only = bits_b - bits_a
        union = bits_a | bits_b
        score = len(shared) / len(union) if union else 1.0
        display(Math(r'T(A,B)=\frac{|F_A\cap F_B|}{|F_A\cup F_B|}' +
                     rf'=\frac{{{len(shared)}}}{{{len(union)}}}={score:.3f}'))
        print(f'A only: {len(a_only)} bits. Both: {len(shared)} bits. B only: {len(b_only)} bits.')
        print(f'Union: {len(a_only)} + {len(shared)} + {len(b_only)} = {len(union)} bits.')
        fig, ax = plt.subplots(figsize=(8, 3.2))
        left = 0
        groups = ((len(a_only),'#4477aa','Only A'),
                  (len(shared),'#2b9b8a','Both'),
                  (len(b_only),'#d89922','Only B'))
        for count, colour, label in groups:
            ax.barh(0, count, left=left, color=colour, label=label, height=0.48)
            if count:
                ax.text(left + count/2, 0, str(count), ha='center', va='center',
                        color='white' if count > 2 else '#172b3a', fontweight='bold')
            left += count
        ax.set_xlim(0, max(1, len(union))*1.03)
        ax.set_yticks([])
        ax.set_xlabel('Number of fingerprint bits')
        ax.set_title('Bits in either fingerprint')
        handles, labels = ax.get_legend_handles_labels()
        fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, 0.03),
                   ncol=3, frameon=False)
        fig.subplots_adjust(left=0.07, right=0.98, top=0.79, bottom=0.39)
        plt.show()
        if show_bit_numbers.value:
            print('These numbers are fingerprint slots from 0 to 2,047, not atom numbers.')
            print('Only A slots:', sorted(a_only))
            print('Shared slots:', sorted(shared))
            print('Only B slots:', sorted(b_only))

for control in (pair_a, pair_b, show_bit_numbers):
    control.observe(show_tanimoto_calculation, names='value')
display(widgets.HBox([pair_a, pair_b]), show_bit_numbers, calculation_output)
show_tanimoto_calculation()

---

## 2 · Build and inspect a 3D pharmacophore

A pharmacophore is a set of **chemical features** shared in 3D by molecules believed to have similar activity. A donor can supply a hydrogen bond, an acceptor can receive one, and an aromatic or hydrophobic feature describes a different kind of contact. RDKit makes several 3D conformers for each 2D ligand and optimises them with a molecular mechanics force field. Supplied 3D coordinates are preserved. Its O3A method overlays each ligand on the first one using chemical properties and shape. We then look for features of the same family within a chosen distance. The **support** is how many training ligands contribute to a feature.

Change the distance tolerance and minimum support to see which features remain. This is a simple reference anchored consensus, so the first ligand matters. A pharmacophore normally needs several features in distinct positions. A strong 2D Tanimoto score does not guarantee a good 3D feature match, and a low score does not rule one out. The overlay is a hypothesis because free ligand conformers may differ from bound conformers.

In [ ]:
# @title Step 7. Build the pharmacophore { display-mode: "form" }
feature_tolerance_A = 1.5 # @param {type:'number'}
minimum_support = 3 # @param {type:'integer'}
conformers_per_ligand = 8 # @param {type:'integer'}
if not 0.5 <= feature_tolerance_A <= 3.0:
    raise ValueError('Choose a feature tolerance from 0.5 to 3.0 Å')
if not 2 <= minimum_support <= len(training):
    raise ValueError('Minimum support must be from 2 to the number of training ligands')
if not 1 <= conformers_per_ligand <= 30:
    raise ValueError('Choose 1 to 30 conformers per ligand')
reference_options, reference_source = make_conformers(training[0][1], count=conformers_per_ligand)
reference = reference_options[0]
aligned = [(training[0][0], reference)]
alignment_rows = [{'ligand':training[0][0], 'source':reference_source, 'O3A score':'Reference'}]
for name, mol in training[1:]:
    fitted, score, source = align_best(mol, reference, count=conformers_per_ligand)
    aligned.append((name, fitted))
    alignment_rows.append({'ligand':name, 'source':source, 'O3A score':score})
pharmacophore = consensus_features(aligned, feature_tolerance_A, minimum_support)
if not pharmacophore:
    raise ValueError('No shared features at these settings. Try a larger tolerance or lower minimum support.')
from IPython.display import display
import pandas as pd
display(pd.DataFrame(alignment_rows))
feature_rows = [{'feature':i, 'family':f['family'], 'support':f"{f['support']}/{len(aligned)}",
                 'spread Å':round(f['spread'],2), 'x Å':round(f['xyz'][0],2),
                 'y Å':round(f['xyz'][1],2), 'z Å':round(f['xyz'][2],2)}
                for i, f in enumerate(pharmacophore,1)]
display(pd.DataFrame(feature_rows))
if len(pharmacophore) < 3:
    print('Only a few features were found. Try other training ligands or settings before screening.')
print('Pharmacophore ready:', len(pharmacophore), 'shared features.')

In [ ]:
# @title Step 8. Explore the 3D pharmacophore { display-mode: "form" }
ligand_colours = ['#3949ab','#00897b','#e67e22','#8e44ad','#c0392b','#54747c']
visible_ligands = widgets.SelectMultiple(options=[name for name, _ in aligned],
                                         value=tuple(name for name, _ in aligned),
                                         description='Ligands', rows=min(8,len(aligned)),
                                         layout=widgets.Layout(width='320px'))
show_features = widgets.Checkbox(value=True, description='Show feature spheres')
representation = widgets.Dropdown(options=['Sticks','Lines','Ball and stick'],
                                  value='Sticks', description='Ligands')
viewer_output = Output()

def add_snapshot_button(view, filename):
    view.startjs += f'''var box = document.getElementById('3dmolviewer_UNIQUEID');
var button = document.createElement('button');
button.textContent = 'Save current view as PNG';
button.style.cssText = 'display:block;margin:8px 0;padding:9px 14px;background:#145a82;color:white;border:0;border-radius:6px;cursor:pointer;font-weight:600';
button.onclick = function() {{ var link = document.createElement('a'); link.href = viewer_UNIQUEID.pngURI(); link.download = {json.dumps(filename)}; link.click(); }};
box.parentNode.insertBefore(button, box);
'''

def show_overlay(change=None):
    with viewer_output:
        viewer_output.clear_output(wait=True)
        view = py3Dmol.view(width=850, height=550)
        for index, (name, mol) in enumerate(aligned):
            if name not in visible_ligands.value:
                continue
            view.addModel(Chem.MolToMolBlock(mol), 'sdf')
            model = view.getModel()
            colour = ligand_colours[index % len(ligand_colours)]
            if representation.value == 'Lines':
                model.setStyle({'line':{'color':colour,'linewidth':2}})
            elif representation.value == 'Ball and stick':
                model.setStyle({'stick':{'color':colour,'radius':0.13},
                                'sphere':{'color':colour,'scale':0.22}})
            else:
                model.setStyle({'stick':{'color':colour,'radius':0.14}})
        if show_features.value:
            for index, f in enumerate(pharmacophore,1):
                xyz = dict(zip('xyz', map(float,f['xyz'])))
                colour = COLOURS[f['family']]
                view.addSphere({'center':xyz,'radius':0.6,'color':colour,'opacity':0.55})
                view.addLabel(str(index), {'position':xyz,'backgroundColor':colour,
                                           'fontColor':'white','fontSize':12,'inFront':True})
        view.setBackgroundColor('white')
        view.zoomTo()
        add_snapshot_button(view, 'ligandlab_pharmacophore.png')
        view.show()

for control in (visible_ligands, show_features, representation):
    control.observe(show_overlay, names='value')
display(widgets.HBox([visible_ligands, widgets.VBox([representation, show_features])]), viewer_output)
display(HTML('<b>Feature colours:</b> ' + ' · '.join(
    f'<span style="color:{COLOURS[family]}">●</span> {family}' for family in FAMILIES)))
show_overlay()

---

## 3 · Screen a small library

The demonstration library includes related and unrelated molecules for comparison. You can instead upload a `.smi`, `.csv`, `.sdf`, `.mol` or `.mol2` file. Each candidate is given 3D conformers, overlaid on the reference, then scored by how many pharmacophore features of the correct family lie within the tolerance. The best conformer is retained. The table also shows 2D Tanimoto similarity to the first training ligand. Compare that with the 3D feature match. This takes a little time.

**Match fraction** is the number of matched pharmacophore features divided by the total. The distance column is the average distance for matched features. A high rank is a reason to inspect a candidate, not a prediction of potency. This simple screen has no receptor, steric exclusion volumes, solvation or activity model.

In [ ]:
# @title Step 9. Screen a library { display-mode: "form" }
screen_source = 'Demo library' # @param ['Demo library', 'Upload file']
screen_conformers = 8 # @param {type:'integer'}
if not 1 <= screen_conformers <= 30:
    raise ValueError('Choose 1 to 30 conformers per candidate')
if screen_source == 'Demo library':
    candidates = [labelled_molecule(name, Chem.MolFromSmiles(smiles)) for name, smiles in DEMO_LIBRARY]
else:
    from google.colab import files
    print('Upload one or more candidate files in SDF, MOL, MOL2, SMI or CSV format')
    uploaded = files.upload()
    if not uploaded:
        raise ValueError('Upload at least one candidate file')
    candidates = [item for filename, data in uploaded.items()
                  for item in read_molecules(filename, data)]
if not 1 <= len(candidates) <= 50:
    raise ValueError('Use 1 to 50 candidates for this Colab practical')
if len({name for name, _ in candidates}) != len(candidates):
    raise ValueError('Give each screening candidate a different name')

def match_pharmacophore(mol):
    candidate_features = features(mol)
    possible = sorted((float(np.linalg.norm(item['xyz']-target['xyz'])), ti, ci)
                      for ti, target in enumerate(pharmacophore)
                      for ci, item in enumerate(candidate_features)
                      if item['family'] == target['family'])
    distances, used_targets, used_candidates = [], set(), set()
    for distance, ti, ci in possible:
        if distance > feature_tolerance_A:
            break
        if ti not in used_targets and ci not in used_candidates:
            distances.append(distance)
            used_targets.add(ti)
            used_candidates.add(ci)
    return len(distances), float(np.mean(distances)) if distances else float('inf')

ranked = []
hits = []
failed = []
for name, mol in candidates:
    try:
        conformers, source = make_conformers(mol, count=screen_conformers, seed=23)
        best = None
        for conformer in conformers:
            alignment = rdMolAlign.GetO3A(conformer, reference)
            o3a = float(alignment.Score())
            alignment.Align()
            matched, mean_distance = match_pharmacophore(conformer)
            score = (matched, -mean_distance, o3a)
            if best is None or score > best[0]:
                best = (score, conformer, matched, mean_distance, o3a)
        if best is None:
            raise ValueError('No conformer could be aligned')
        _, pose, matched, mean_distance, o3a = best
        fraction = matched / len(pharmacophore)
        tanimoto = DataStructs.TanimotoSimilarity(training_fingerprints[0], fingerprint(mol))
        ranked.append({'name':name, '2D Tanimoto to reference':round(tanimoto,3), 'matched features':matched,
                       'total features':len(pharmacophore), 'match fraction':round(fraction,3),
                       'mean distance Å':round(mean_distance,2) if matched else '',
                       'O3A score':round(o3a,2), 'SMILES':Chem.MolToSmiles(Chem.RemoveHs(mol))})
        hits.append((name, pose))
    except Exception as error:
        failed.append((name, str(error)))
order = sorted(range(len(ranked)), key=lambda i:(-ranked[i]['match fraction'],
                                                    ranked[i]['mean distance Å'] if ranked[i]['mean distance Å'] != '' else 999,
                                                    -ranked[i]['O3A score']))
ranked = [ranked[i] for i in order]
hits = [hits[i] for i in order]
display(pd.DataFrame(ranked))
print(f'{len(ranked)} candidates ranked. {len(failed)} could not be processed.')
if failed:
    print('Failed candidates:', failed)
if hits:
    hit_picker = widgets.Dropdown(options=[name for name, _ in hits], description='Candidate')
    hit_output = Output()
    def show_hit(change=None):
        with hit_output:
            hit_output.clear_output(wait=True)
            index = [name for name, _ in hits].index(hit_picker.value)
            view = py3Dmol.view(width=760, height=470)
            view.addModel(Chem.MolToMolBlock(reference), 'sdf')
            view.setStyle({'model':0}, {'stick':{'color':'#aeb9c8','radius':0.12}})
            view.addModel(Chem.MolToMolBlock(hits[index][1]), 'sdf')
            view.setStyle({'model':1}, {'stick':{'color':'#167b83','radius':0.16}})
            for f in pharmacophore:
                view.addSphere({'center':dict(zip('xyz',map(float,f['xyz']))),
                                'radius':0.55,'color':COLOURS[f['family']],'opacity':0.5})
            view.setBackgroundColor('white')
            view.zoomTo()
            add_snapshot_button(view, 'ligandlab_candidate.png')
            view.show()
            print('Grey: reference ligand. Teal: selected candidate.')
    hit_picker.observe(show_hit, names='value')
    display(hit_picker, hit_output)
    show_hit()

In [ ]:
# @title Step 10. Download results { display-mode: "form" }
with (WORK/'aligned_training.sdf').open('w') as handle:
    writer = Chem.SDWriter(handle)
    for name, mol in aligned:
        mol.SetProp('_Name', name)
        writer.write(mol)
    writer.close()
with (WORK/'ranked_candidates.sdf').open('w') as handle:
    writer = Chem.SDWriter(handle)
    for row, (name, mol) in zip(ranked, hits):
        mol.SetProp('_Name', name)
        mol.SetProp('match_fraction', str(row['match fraction']))
        writer.write(mol)
    writer.close()
pd.DataFrame(feature_rows).to_csv(WORK/'pharmacophore_features.csv', index=False)
similarity_table.to_csv(WORK/'training_similarity_matrix.csv')
pd.DataFrame(similarity_pairs).to_csv(WORK/'training_similarity_pairs.csv', index=False)
pd.DataFrame(ranked).to_csv(WORK/'screening_results.csv', index=False)
pd.DataFrame(alignment_rows).to_csv(WORK/'alignment_summary.csv', index=False)
bundle = WORK/'ligandlab_results.zip'
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
    for filename in ('aligned_training.sdf','ranked_candidates.sdf','pharmacophore_features.csv',
                     'tanimoto_heatmap.png',
                     'training_similarity_matrix.csv','training_similarity_pairs.csv',
                     'screening_results.csv','alignment_summary.csv'):
        archive.write(WORK/filename, filename)
print('Ready:', bundle)
from google.colab import files
files.download(str(bundle))

### Questions to discuss

1. Which pair has the highest 2D Tanimoto score, and which atom environments might explain it?
2. Does the screening order by 2D Tanimoto match the order by 3D feature matches? Give an example.
3. Which shared features survive when you increase minimum support to all training ligands?
4. What changes when the distance tolerance is made tighter or looser?
5. Why can a candidate match a pharmacophore yet fail to bind the target?

### References

1. Schneidman-Duhovny D, Dror O, Inbar Y, Nussinov R, Wolfson HJ. PharmaGist: a webserver for ligand-based pharmacophore detection. *Nucleic Acids Res.* 2008, 36:W223–W228. [doi:10.1093/nar/gkn187](https://doi.org/10.1093/nar/gkn187).
2. Landrum G and RDKit contributors. *RDKit: Open-source cheminformatics* [software]. [doi:10.5281/zenodo.591637](https://doi.org/10.5281/zenodo.591637).
3. Rego N, Koes D. 3Dmol.js: molecular visualization with WebGL. *Bioinformatics.* 2015, 31:1322–1324. [doi:10.1093/bioinformatics/btu829](https://doi.org/10.1093/bioinformatics/btu829).
4. EPAM Systems. *Ketcher: web based molecule sketcher* [software]. [Project page](https://github.com/epam/ketcher).